# Historical research notebook — not the canonical final Pure implementation

This notebook is retained for research provenance and training logic. Its original Pure post-processing branch predates the final corrected Pure-specific spatial implementation used for the reported manuscript results. It must not be used as the authoritative final Pure scoring path.

For the final reported Full/Pure scoring logic, use `src/proposed_method.py` and `scripts/reproduce_full_pure.py`.


In [ ]:
# =========================
# Reproducibility
# =========================

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Random seed:", SEED)

In [ ]:
import numpy as np
import pandas as pd

# =========================
# 1. Load dataset
# =========================

X_path = "X.npy"
y_path = "y.npy"

X = np.load(X_path)
y = np.load(y_path)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype)
print("y dtype:", y.dtype)

# =========================
# 2. Basic checks
# =========================

assert X.ndim == 2, "X must be a 2D array with shape (samples, features)."
assert y.ndim in (1, 2), "y must be a 1D or 2D array."
assert X.shape[0] == y.shape[0], (
    "X and y must contain the same number of samples."
)

expected_features = 23

assert X.shape[1] == expected_features, (
    f"Expected {expected_features} features, but found {X.shape[1]}."
)

assert np.issubdtype(X.dtype, np.number), (
    "X must contain numerical values."
)

print("\nBasic shape and datatype checks passed.")

# =========================
# 3. Define feature-view groups
# =========================
#
# These definitions correspond to the multi-view architecture used
# in the current methodology. Flow accumulation is intentionally
# included in both the terrain and hydrological views.

feature_groups = {
    "terrain": [
        "elevation",
        "slope",
        "profile_curv",
        "plan_curv",
        "depression_depth",
    ],

    "hydrological": [
        "flow_acc",
        "flow_contrast",
        "river_dist",
    ],

    "spatial": [
        "local_std",
        "elev_percentile",
        "elev_contrast",
        "slope_variability",
    ],

    "infrastructure": [
        "road_dist",
    ],

    "surface": [
        "ndvi",
        "ndwi",
        "ndbi",
    ],

    "landuse": [
        "landuse_1",
        "landuse_2",
        "landuse_3",
        "landuse_4",
        "landuse_5",
        "landuse_6",
        "landuse_7",
    ],
}

print("\nFeature-view groups defined:")
for group_name, group_features in feature_groups.items():
    print(f"{group_name}: {len(group_features)} features")

In [ ]:
# =========================
# 4. Define feature names
# =========================

# The order below must exactly match the column order used when X.npy
# was created in the feature-engineering notebook.

feature_names = [
    # Terrain and hydrological source features
    "elevation",             # 0
    "slope",                 # 1
    "flow_acc",              # 2
    "profile_curv",          # 3
    "plan_curv",             # 4
    "depression_depth",      # 5
    "local_std",             # 6

    # Spatial and local contextual features
    "elev_percentile",       # 7
    "elev_contrast",         # 8
    "slope_variability",     # 9
    "flow_contrast",         # 10

    # Infrastructure and proximity features
    "road_dist",             # 11
    "river_dist",            # 12

    # Surface/spectral features
    "ndvi",                  # 13
    "ndwi",                  # 14
    "ndbi",                  # 15

    # One-hot-encoded land-use features
    "landuse_1",             # 16
    "landuse_2",             # 17
    "landuse_3",             # 18
    "landuse_4",             # 19
    "landuse_5",             # 20
    "landuse_6",             # 21
    "landuse_7",             # 22
]

# Sanity check
assert len(feature_names) == X.shape[1], (
    f"Feature-name mismatch: {len(feature_names)} names "
    f"for {X.shape[1]} columns."
)

assert len(set(feature_names)) == len(feature_names), (
    "Feature names must be unique."
)

df = pd.DataFrame(X, columns=feature_names)

print("First five rows:")
display(df.head())

print("\nFeature summary:")
display(df.describe().T)

print("\nMissing values per feature:")
display(df.isna().sum().to_frame(name="missing_count"))

print("\nInfinite values per feature:")
infinite_counts = pd.Series(
    np.isinf(X).sum(axis=0),
    index=feature_names,
    name="infinite_count",
)
display(infinite_counts.to_frame())

# Dataset-wide validation
total_missing = int(np.isnan(X).sum())
total_infinite = int(np.isinf(X).sum())

print("\nTotal missing values:", total_missing)
print("Total infinite values:", total_infinite)

assert total_missing == 0, (
    "X contains missing values. Resolve them before model training."
)

assert total_infinite == 0, (
    "X contains infinite values. Resolve them before model training."
)

print("\nFeature-name and numerical-value checks passed.")

In [ ]:
# =========================
# 5. Validate land-use one-hot encoding
# =========================

landuse_cols = [
    "landuse_1",
    "landuse_2",
    "landuse_3",
    "landuse_4",
    "landuse_5",
    "landuse_6",
    "landuse_7",
]

landuse_data = df[landuse_cols]

# -------------------------
# Check column values
# -------------------------

print("Land-use unique values per column:")

non_binary_columns = []

for col in landuse_cols:
    unique_values = np.sort(landuse_data[col].unique())
    print(f"{col}: {unique_values}")

    if not np.all(np.isin(unique_values, [0, 1])):
        non_binary_columns.append(col)

if non_binary_columns:
    print(
        "\nNon-binary values were found in:",
        non_binary_columns
    )
else:
    print("\nAll land-use columns contain only binary values.")

# -------------------------
# Check row-wise one-hot sums
# -------------------------

landuse_sum = landuse_data.sum(axis=1).to_numpy()

valid_mask_landuse = np.isclose(landuse_sum, 1.0)
zero_mask_landuse = np.isclose(landuse_sum, 0.0)
multiple_mask_landuse = landuse_sum > 1.0
other_invalid_mask = ~(
    valid_mask_landuse
    | zero_mask_landuse
    | multiple_mask_landuse
)

valid_rows = np.flatnonzero(valid_mask_landuse)
zero_rows = np.flatnonzero(zero_mask_landuse)
multiple_rows = np.flatnonzero(multiple_mask_landuse)
other_invalid_rows = np.flatnonzero(other_invalid_mask)

print("\nLand-use row-sum counts:")
print("Valid one-hot rows (sum = 1):", valid_rows.size)
print("Missing land-use rows (sum = 0):", zero_rows.size)
print("Multiple-class rows (sum > 1):", multiple_rows.size)
print("Other invalid rows:", other_invalid_rows.size)

# Show examples only when problems exist.
if multiple_rows.size > 0:
    print("\nRows containing multiple active land-use classes:")
    display(
        df.iloc[multiple_rows[:10]][landuse_cols]
    )

if zero_rows.size > 0:
    print("\nRows containing no active land-use class:")
    display(
        df.iloc[zero_rows[:10]][landuse_cols]
    )

if other_invalid_rows.size > 0:
    print("\nRows with unexpected land-use sums:")
    display(
        df.iloc[other_invalid_rows[:10]][landuse_cols]
    )

# Final validation
assert len(non_binary_columns) == 0, (
    "One or more land-use columns contain values other than 0 and 1."
)

assert multiple_rows.size == 0, (
    "Some rows contain more than one active land-use class."
)

assert other_invalid_rows.size == 0, (
    "Some rows contain unexpected land-use encoding values."
)

# Based on your feature-engineering pipeline, every retained pixel
# should normally have exactly one land-use class.
assert zero_rows.size == 0, (
    "Some rows contain no active land-use class."
)

print("\nLand-use one-hot encoding is valid.")

In [ ]:
# =========================
# 6. Create disjoint feature views
# =========================

# Terrain: 5 features
terrain_idx = [
    0,  # elevation
    1,  # slope
    3,  # profile_curv
    4,  # plan_curv
    5,  # depression_depth
]

# Hydrological: 3 features
hydro_idx = [
    2,   # flow_acc
    10,  # flow_contrast
    12,  # river_dist
]

# Spatial/contextual: 4 features
spatial_idx = [
    6,  # local_std
    7,  # elev_percentile
    8,  # elev_contrast
    9,  # slope_variability
]

# Infrastructure: 1 feature
infra_idx = [
    11,  # road_dist
]

# Surface/spectral: 3 features
surface_idx = [
    13,  # ndvi
    14,  # ndwi
    15,  # ndbi
]

# Land use: 7 features
landuse_idx = list(range(16, 23))

view_indices = {
    "terrain": terrain_idx,
    "hydrological": hydro_idx,
    "spatial": spatial_idx,
    "infrastructure": infra_idx,
    "surface": surface_idx,
    "landuse": landuse_idx,
}

# Verify that all 23 features occur exactly once.
all_indices = [
    idx
    for indices in view_indices.values()
    for idx in indices
]

assert len(all_indices) == 23
assert len(set(all_indices)) == 23
assert sorted(all_indices) == list(range(23))

# Create views.
X_terrain = X[:, terrain_idx].astype(np.float32, copy=True)
X_hydro = X[:, hydro_idx].astype(np.float32, copy=True)
X_spatial = X[:, spatial_idx].astype(np.float32, copy=True)
X_infra = X[:, infra_idx].astype(np.float32, copy=True)
X_surface = X[:, surface_idx].astype(np.float32, copy=True)
X_landuse = X[:, landuse_idx].astype(np.float32, copy=True)

print("Terrain view shape:", X_terrain.shape)
print("Hydrological view shape:", X_hydro.shape)
print("Spatial view shape:", X_spatial.shape)
print("Infrastructure view shape:", X_infra.shape)
print("Surface view shape:", X_surface.shape)
print("Land-use view shape:", X_landuse.shape)

print("\nFeatures in each view:")
for view_name, indices in view_indices.items():
    names = [feature_names[idx] for idx in indices]
    print(f"{view_name}: {names}")

print("\nAll 23 features are assigned exactly once.")

In [ ]:
# =========================
# 6. Final sanity check for all views
# =========================

views = {
    "terrain": X_terrain,
    "hydrological": X_hydro,
    "spatial": X_spatial,
    "infrastructure": X_infra,
    "surface": X_surface,
    "landuse": X_landuse,
}

expected_dimensions = {
    "terrain": 5,
    "hydrological": 3,
    "spatial": 4,
    "infrastructure": 1,
    "surface": 3,
    "landuse": 7,
}

for name, arr in views.items():
    print(f"\n{name.upper()} VIEW")
    print("Shape:", arr.shape)

    # Check expected number of columns
    assert arr.ndim == 2, (
        f"{name} view must be a two-dimensional array."
    )

    assert arr.shape[0] == X.shape[0], (
        f"{name} view contains an incorrect number of samples."
    )

    assert arr.shape[1] == expected_dimensions[name], (
        f"{name} view should contain "
        f"{expected_dimensions[name]} features, "
        f"but contains {arr.shape[1]}."
    )

    nan_count = int(np.isnan(arr).sum())
    inf_count = int(np.isinf(arr).sum())

    print("NaN count:", nan_count)
    print("Inf count:", inf_count)

    if arr.size > 0 and nan_count < arr.size:
        print("Min:", float(np.nanmin(arr)))
        print("Max:", float(np.nanmax(arr)))
    else:
        print("Min: unavailable")
        print("Max: unavailable")

    assert nan_count == 0, (
        f"{name} view contains NaN values."
    )

    assert inf_count == 0, (
        f"{name} view contains infinite values."
    )

print("\nAll feature-view sanity checks passed.")

In [ ]:
# =========================
# 7. Terrain preprocessing
# =========================

X_terrain_proc = X_terrain.astype(
    np.float32,
    copy=True
)

# Internal terrain-view order:
# 0: elevation
# 1: slope
# 2: profile_curv
# 3: plan_curv
# 4: depression_depth

terrain_clip_bounds = {}

# -------------------------
# 1. Curvature features
# -------------------------

curvature_features = {
    "profile_curv": 2,
    "plan_curv": 3,
}

for feature_name, idx in curvature_features.items():
    curv = X_terrain_proc[:, idx]

    lower = float(np.percentile(curv, 1))
    upper = float(np.percentile(curv, 99))

    terrain_clip_bounds[feature_name] = {
        "lower": lower,
        "upper": upper,
    }

    curv_clipped = np.clip(
        curv,
        lower,
        upper
    )

    # Preserve the transformation used in the current methodology.
    X_terrain_proc[:, idx] = np.tanh(
        curv_clipped
    )

# -------------------------
# 2. Depression depth
# -------------------------

# Protect log1p against unexpected negative numerical values.
X_terrain_proc[:, 4] = np.log1p(
    np.clip(
        X_terrain_proc[:, 4],
        a_min=0.0,
        a_max=None
    )
)

# -------------------------
# 3. Final checks
# -------------------------

assert X_terrain_proc.shape == X_terrain.shape
assert np.isfinite(X_terrain_proc).all()

print("Terrain preprocessing completed.")
print("Terrain processed shape:", X_terrain_proc.shape)
print("Curvature clipping bounds:", terrain_clip_bounds)

In [ ]:
# =========================
# 8. Spatial-feature preprocessing
# =========================

X_spatial_proc = X_spatial.astype(
    np.float32,
    copy=True
)

spatial_feature_names = [
    "local_std",
    "elev_percentile",
    "elev_contrast",
    "slope_variability",
]

spatial_clip_bounds = {}

for idx, feature_name in enumerate(spatial_feature_names):
    col = X_spatial_proc[:, idx]

    lower = float(np.percentile(col, 1))
    upper = float(np.percentile(col, 99))

    spatial_clip_bounds[feature_name] = {
        "lower": lower,
        "upper": upper,
    }

    X_spatial_proc[:, idx] = np.clip(
        col,
        lower,
        upper
    )

assert X_spatial_proc.shape == X_spatial.shape
assert np.isfinite(X_spatial_proc).all()

print("Spatial features clipped.")
print("Spatial processed shape:", X_spatial_proc.shape)
print("Spatial clipping bounds:", spatial_clip_bounds)

In [ ]:
# =========================
# 9. Robust scaling
# =========================

from sklearn.preprocessing import RobustScaler
import joblib

terrain_scaler = RobustScaler()
hydro_scaler = RobustScaler()
spatial_scaler = RobustScaler()
surface_scaler = RobustScaler()
infra_scaler = RobustScaler()

# -------------------------
# Fit and transform views
# -------------------------

X_terrain_scaled = terrain_scaler.fit_transform(
    X_terrain_proc
).astype(np.float32)

X_hydro_scaled = hydro_scaler.fit_transform(
    X_hydro
).astype(np.float32)

X_spatial_scaled = spatial_scaler.fit_transform(
    X_spatial_proc
).astype(np.float32)

X_surface_scaled = surface_scaler.fit_transform(
    X_surface
).astype(np.float32)

X_infra_scaled = infra_scaler.fit_transform(
    X_infra
).astype(np.float32)

# Land-use indicators remain binary and are not scaled.
X_landuse_scaled = X_landuse.astype(
    np.float32,
    copy=True
)

# -------------------------
# Save fitted preprocessing
# -------------------------

joblib.dump(
    terrain_scaler,
    "terrain_scaler.pkl"
)

joblib.dump(
    hydro_scaler,
    "hydro_scaler.pkl"
)

joblib.dump(
    spatial_scaler,
    "spatial_scaler.pkl"
)

joblib.dump(
    surface_scaler,
    "surface_scaler.pkl"
)

joblib.dump(
    infra_scaler,
    "infra_scaler.pkl"
)

joblib.dump(
    terrain_clip_bounds,
    "terrain_clip_bounds.pkl"
)

joblib.dump(
    spatial_clip_bounds,
    "spatial_clip_bounds.pkl"
)

# -------------------------
# Final checks and summaries
# -------------------------

scaled_views = {
    "terrain": X_terrain_scaled,
    "hydrological": X_hydro_scaled,
    "spatial": X_spatial_scaled,
    "surface": X_surface_scaled,
    "infrastructure": X_infra_scaled,
    "landuse": X_landuse_scaled,
}

for name, arr in scaled_views.items():
    assert np.isfinite(arr).all(), (
        f"{name} scaled view contains NaN or infinite values."
    )

    print(
        f"{name.capitalize()} shape: {arr.shape}, "
        f"min: {arr.min():.6f}, "
        f"max: {arr.max():.6f}, "
        f"median: {np.median(arr):.6f}"
    )

print("\nRobust scaling completed and preprocessing objects saved.")

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, Subset, random_split
from tqdm import tqdm

# =========================
# Reproducibility
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# =========================
# Device setup
# =========================

if torch.cuda.is_available():
    device = torch.device("cuda")

    # Improve reproducibility on CUDA.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")

else:
    device = torch.device("cpu")

print("Using device:", device)
print("Random seed:", SEED)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =========================
# Step 2: Ensure float32 representation
# =========================

X_terrain_scaled = np.asarray(
    X_terrain_scaled,
    dtype=np.float32
)

X_hydro_scaled = np.asarray(
    X_hydro_scaled,
    dtype=np.float32
)

X_spatial_scaled = np.asarray(
    X_spatial_scaled,
    dtype=np.float32
)

X_surface_scaled = np.asarray(
    X_surface_scaled,
    dtype=np.float32
)

X_infra_scaled = np.asarray(
    X_infra_scaled,
    dtype=np.float32
)

X_landuse_scaled = np.asarray(
    X_landuse_scaled,
    dtype=np.float32
)

# -------------------------
# Validate shapes
# -------------------------

expected_shapes = {
    "Terrain": (X.shape[0], 5),
    "Hydrological": (X.shape[0], 3),
    "Spatial": (X.shape[0], 4),
    "Surface": (X.shape[0], 3),
    "Infrastructure": (X.shape[0], 1),
    "Landuse": (X.shape[0], 7),
}

scaled_views = {
    "Terrain": X_terrain_scaled,
    "Hydrological": X_hydro_scaled,
    "Spatial": X_spatial_scaled,
    "Surface": X_surface_scaled,
    "Infrastructure": X_infra_scaled,
    "Landuse": X_landuse_scaled,
}

for name, arr in scaled_views.items():
    assert arr.shape == expected_shapes[name], (
        f"{name} shape mismatch: expected "
        f"{expected_shapes[name]}, found {arr.shape}."
    )

    assert arr.dtype == np.float32, (
        f"{name} must have dtype float32."
    )

    assert np.isfinite(arr).all(), (
        f"{name} contains NaN or infinite values."
    )

    print(
        f"{name}: shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

print("\nAll scaled views are valid.")

In [ ]:
# =========================
# Step 3: Create full processed input matrix
# =========================

# The six views are disjoint and jointly cover all 23 features:
# 5 + 3 + 4 + 3 + 1 + 7 = 23.

X_full_scaled = np.concatenate(
    [
        X_terrain_scaled,   # 5 features
        X_hydro_scaled,     # 3 features
        X_spatial_scaled,   # 4 features
        X_surface_scaled,   # 3 features
        X_infra_scaled,     # 1 feature
        X_landuse_scaled,   # 7 features
    ],
    axis=1
).astype(np.float32, copy=False)

# -------------------------
# Final validation
# -------------------------

expected_full_shape = (
    X.shape[0],
    23
)

assert X_full_scaled.shape == expected_full_shape, (
    f"Expected full input shape {expected_full_shape}, "
    f"but found {X_full_scaled.shape}."
)

assert X_full_scaled.dtype == np.float32, (
    "X_full_scaled must have dtype float32."
)

assert np.isfinite(X_full_scaled).all(), (
    "X_full_scaled contains NaN or infinite values."
)

print("Full scaled input shape:", X_full_scaled.shape)
print("Full scaled input dtype:", X_full_scaled.dtype)
print(
    "Full scaled input min/max:",
    float(X_full_scaled.min()),
    float(X_full_scaled.max())
)

print("\nFull processed input matrix created successfully.")

In [ ]:
# =========================
# Step 4: Create PyTorch Dataset
# =========================

class MultiViewGeoDataset(Dataset):
    def __init__(
        self,
        X_terrain,
        X_hydro,
        X_spatial,
        X_surface,
        X_infra,
        X_landuse,
        X_full
    ):
        # Store the NumPy arrays without creating additional copies.
        self.X_terrain = X_terrain
        self.X_hydro = X_hydro
        self.X_spatial = X_spatial
        self.X_surface = X_surface
        self.X_infra = X_infra
        self.X_landuse = X_landuse
        self.X_full = X_full

        # -------------------------
        # Validate sample counts
        # -------------------------

        n_samples = X_full.shape[0]

        arrays = {
            "terrain": X_terrain,
            "hydrological": X_hydro,
            "spatial": X_spatial,
            "surface": X_surface,
            "infrastructure": X_infra,
            "landuse": X_landuse,
            "full": X_full,
        }

        for name, arr in arrays.items():
            assert arr.ndim == 2, (
                f"{name} array must be two-dimensional."
            )

            assert arr.shape[0] == n_samples, (
                f"{name} contains {arr.shape[0]} samples, "
                f"but X_full contains {n_samples}."
            )

            assert arr.dtype == np.float32, (
                f"{name} must use float32 values."
            )

            assert np.isfinite(arr).all(), (
                f"{name} contains NaN or infinite values."
            )

    def __len__(self):
        return self.X_full.shape[0]

    def __getitem__(self, idx):
        terrain = torch.from_numpy(self.X_terrain[idx])
        hydro = torch.from_numpy(self.X_hydro[idx])
        spatial = torch.from_numpy(self.X_spatial[idx])
        surface = torch.from_numpy(self.X_surface[idx])
        infra = torch.from_numpy(self.X_infra[idx])
        landuse = torch.from_numpy(self.X_landuse[idx])
        full = torch.from_numpy(self.X_full[idx])

        return (
            terrain,
            hydro,
            spatial,
            surface,
            infra,
            landuse,
            full,
        )

In [ ]:
# =========================
# Step 5: Build dataset and train-validation split
# =========================

dataset = MultiViewGeoDataset(
    X_terrain_scaled,
    X_hydro_scaled,
    X_spatial_scaled,
    X_surface_scaled,
    X_infra_scaled,
    X_landuse_scaled,
    X_full_scaled
)

total_size = len(dataset)
val_size = int(0.10 * total_size)
train_size = total_size - val_size

split_generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=split_generator
)

print("Total samples:", total_size)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

assert len(train_dataset) + len(val_dataset) == total_size
assert len(train_dataset) == train_size
assert len(val_dataset) == val_size

print("\nDataset split completed.")

In [ ]:
# =========================
# Step 6: Create DataLoaders
# =========================

batch_size = 4096

use_pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=use_pin_memory,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=use_pin_memory,
    drop_last=False
)

print("DataLoaders created.")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", batch_size)
print("Pin memory:", use_pin_memory)

In [ ]:
# =========================
# Step 7: Multi-view autoencoder
# =========================

class MultiViewAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # -------------------------
        # View-specific encoders
        # -------------------------

        # Terrain: 5 -> 32 -> 16
        self.terrain_encoder = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Hydrological: 3 -> 16 -> 8
        self.hydro_encoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Spatial: 4 -> 16 -> 8
        self.spatial_encoder = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Surface: 3 -> 16 -> 8
        self.surface_encoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Infrastructure: 1 -> 8 -> 4
        self.infra_encoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU()
        )

        # Land use: 7 -> 16 -> 8
        self.landuse_encoder = nn.Sequential(
            nn.Linear(7, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # -------------------------
        # Fusion network
        # -------------------------

        # Combined view representation:
        # 16 + 8 + 8 + 8 + 4 + 8 = 52
        self.fusion = nn.Sequential(
            nn.Linear(52, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # -------------------------
        # Decoder
        # -------------------------

        # Reconstruct the complete 23-dimensional processed vector.
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 23)
        )

    def forward(
        self,
        terrain,
        hydro,
        spatial,
        surface,
        infra,
        landuse
    ):
        e_terrain = self.terrain_encoder(terrain)
        e_hydro = self.hydro_encoder(hydro)
        e_spatial = self.spatial_encoder(spatial)
        e_surface = self.surface_encoder(surface)
        e_infra = self.infra_encoder(infra)
        e_landuse = self.landuse_encoder(landuse)

        combined = torch.cat(
            [
                e_terrain,
                e_hydro,
                e_spatial,
                e_surface,
                e_infra,
                e_landuse,
            ],
            dim=1
        )

        embedding = self.fusion(combined)
        reconstruction = self.decoder(embedding)

        return reconstruction, embedding

In [ ]:
# =========================
# Step 8: Initialize model, loss, and optimizer
# =========================

model = MultiViewAutoencoder().to(device)

criterion = nn.MSELoss(
    reduction="mean"
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

max_grad_norm = 5.0

print(model)

# Optional parameter summary
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("\nTotal parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

In [ ]:
# =========================
# Step 9: Training function
# =========================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    max_grad_norm=5.0
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for (
        terrain,
        hydro,
        spatial,
        surface,
        infra,
        landuse,
        full
    ) in progress_bar:

        terrain = terrain.to(
            device,
            non_blocking=True
        )
        hydro = hydro.to(
            device,
            non_blocking=True
        )
        spatial = spatial.to(
            device,
            non_blocking=True
        )
        surface = surface.to(
            device,
            non_blocking=True
        )
        infra = infra.to(
            device,
            non_blocking=True
        )
        landuse = landuse.to(
            device,
            non_blocking=True
        )
        full = full.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        reconstruction, _ = model(
            terrain,
            hydro,
            spatial,
            surface,
            infra,
            landuse
        )

        loss = criterion(
            reconstruction,
            full
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_grad_norm
        )

        optimizer.step()

        current_batch_size = full.size(0)

        total_loss += (
            loss.item()
            * current_batch_size
        )

        total_samples += current_batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.6f}"
        )

    if total_samples == 0:
        raise RuntimeError(
            "The training DataLoader returned no samples."
        )

    return total_loss / total_samples

In [ ]:
# =========================
# Step 10: Validation function
# =========================

def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():
        for (
            terrain,
            hydro,
            spatial,
            surface,
            infra,
            landuse,
            full
        ) in progress_bar:

            terrain = terrain.to(
                device,
                non_blocking=True
            )
            hydro = hydro.to(
                device,
                non_blocking=True
            )
            spatial = spatial.to(
                device,
                non_blocking=True
            )
            surface = surface.to(
                device,
                non_blocking=True
            )
            infra = infra.to(
                device,
                non_blocking=True
            )
            landuse = landuse.to(
                device,
                non_blocking=True
            )
            full = full.to(
                device,
                non_blocking=True
            )

            reconstruction, _ = model(
                terrain,
                hydro,
                spatial,
                surface,
                infra,
                landuse
            )

            loss = criterion(
                reconstruction,
                full
            )

            current_batch_size = full.size(0)

            total_loss += (
                loss.item()
                * current_batch_size
            )

            total_samples += current_batch_size

            progress_bar.set_postfix(
                loss=f"{loss.item():.6f}"
            )

    if total_samples == 0:
        raise RuntimeError(
            "The validation DataLoader returned no samples."
        )

    return total_loss / total_samples

In [ ]:
# =========================
# Step 11: Train the model
# =========================

num_epochs = 10
model_path = "multiview_autoencoder_best.pth"

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(num_epochs):
    print(
        f"\nEpoch {epoch + 1}/{num_epochs}"
    )

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        max_grad_norm=max_grad_norm
    )

    val_loss = validate_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Train Loss: {train_loss:.6f}"
    )
    print(
        f"Val Loss:   {val_loss:.6f}"
    )

    # Save the checkpoint with the lowest validation loss.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            model_path
        )

        print(
            "Best validation checkpoint saved."
        )

    elif epoch > 0:
        print(
            "Validation loss did not improve."
        )

# -------------------------
# Reload best checkpoint
# -------------------------

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device
    )
)

model.eval()

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    f"{best_val_loss:.6f}"
)
print(
    "Best model loaded from:",
    model_path
)

# Save training history for later analysis.
np.save(
    "multiview_train_losses.npy",
    np.asarray(
        train_losses,
        dtype=np.float32
    )
)

np.save(
    "multiview_validation_losses.npy",
    np.asarray(
        val_losses,
        dtype=np.float32
    )
)

print("Training history saved.")

In [ ]:
# =========================
# Plot training history
# =========================

epochs = np.arange(
    1,
    len(train_losses) + 1
)

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    epochs,
    train_losses,
    marker="o",
    label="Training loss"
)

plt.plot(
    epochs,
    val_losses,
    marker="s",
    label="Validation loss"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch ({best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("Mean squared reconstruction loss")
plt.title("Multi-View Autoencoder Training Curve")
plt.xticks(epochs)
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    "multiview_autoencoder_training_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================
# Step 13: Extract embeddings and reconstruction errors
# =========================

full_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

model.eval()

n_samples = len(dataset)
embedding_dim = 16

# Preallocate final outputs.
embeddings = np.empty(
    (n_samples, embedding_dim),
    dtype=np.float32
)

embedding_error = np.empty(
    n_samples,
    dtype=np.float32
)

start_idx = 0

with torch.no_grad():
    for (
        terrain,
        hydro,
        spatial,
        surface,
        infra,
        landuse,
        full
    ) in tqdm(
        full_loader,
        desc="Extracting embeddings"
    ):

        terrain = terrain.to(
            device,
            non_blocking=True
        )
        hydro = hydro.to(
            device,
            non_blocking=True
        )
        spatial = spatial.to(
            device,
            non_blocking=True
        )
        surface = surface.to(
            device,
            non_blocking=True
        )
        infra = infra.to(
            device,
            non_blocking=True
        )
        landuse = landuse.to(
            device,
            non_blocking=True
        )
        full = full.to(
            device,
            non_blocking=True
        )

        reconstruction, embedding = model(
            terrain,
            hydro,
            spatial,
            surface,
            infra,
            landuse
        )

        # Per-pixel mean squared reconstruction error.
        error = torch.mean(
            (reconstruction - full) ** 2,
            dim=1
        )

        embedding_np = embedding.cpu().numpy().astype(
            np.float32,
            copy=False
        )

        error_np = error.cpu().numpy().astype(
            np.float32,
            copy=False
        )

        current_batch_size = embedding_np.shape[0]
        end_idx = start_idx + current_batch_size

        embeddings[start_idx:end_idx] = embedding_np
        embedding_error[start_idx:end_idx] = error_np

        start_idx = end_idx

assert start_idx == n_samples
assert embeddings.shape == (n_samples, 16)
assert embedding_error.shape == (n_samples,)
assert np.isfinite(embeddings).all()
assert np.isfinite(embedding_error).all()

print("Embeddings shape:", embeddings.shape)
print("Reconstruction-error shape:", embedding_error.shape)
print("Embeddings dtype:", embeddings.dtype)
print("Reconstruction-error dtype:", embedding_error.dtype)

# These two files are required later.
np.save(
    "multiview_embeddings.npy",
    embeddings
)

np.save(
    "ae_reconstruction_error.npy",
    embedding_error
)

print("\nSaved:")
print("multiview_embeddings.npy")
print("ae_reconstruction_error.npy")

In [ ]:
# =========================
# Step 14: Reconstruction-error summary
# =========================

error_percentiles = np.percentile(
    embedding_error,
    [90, 95, 98, 99]
)

print("Autoencoder reconstruction-error summary:")
print("Minimum:", float(embedding_error.min()))
print("Maximum:", float(embedding_error.max()))
print("Mean:", float(embedding_error.mean()))
print("Median:", float(np.median(embedding_error)))
print("90th percentile:", float(error_percentiles[0]))
print("95th percentile:", float(error_percentiles[1]))
print("98th percentile:", float(error_percentiles[2]))
print("99th percentile:", float(error_percentiles[3]))

In [ ]:
# =========================
# Step 15: Plot reconstruction-error distribution
# =========================

plt.figure(figsize=(7, 5))

plt.hist(
    embedding_error,
    bins=100
)

percentiles_to_plot = {
    95: np.percentile(embedding_error, 95),
    98: np.percentile(embedding_error, 98),
    99: np.percentile(embedding_error, 99),
}

for percentile, value in percentiles_to_plot.items():
    plt.axvline(
        value,
        linestyle="--",
        label=f"{percentile}th percentile: {value:.4f}"
    )

plt.xlabel("Mean Squared Reconstruction Error")
plt.ylabel("Number of Pixels")
plt.title("Distribution of Autoencoder Reconstruction Errors")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Hybrid anomaly detection stage: This is where our method becomes novel.

In [ ]:
# =========================
# Step 16: Normalize reconstruction error
# =========================

# Robust limits calculated from the original dataset.
ae_p1 = float(
    np.percentile(
        embedding_error,
        1
    )
)

ae_p99 = float(
    np.percentile(
        embedding_error,
        99
    )
)

if ae_p99 <= ae_p1:
    raise ValueError(
        "Invalid normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

embedding_error_clipped = np.clip(
    embedding_error,
    ae_p1,
    ae_p99
)

ae_score_norm = (
    embedding_error_clipped - ae_p1
) / (
    ae_p99 - ae_p1
)

ae_score_norm = ae_score_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(ae_score_norm).all()
assert ae_score_norm.min() >= 0.0
assert ae_score_norm.max() <= 1.0

print("AE-score normalization completed.")
print("1st-percentile bound:", ae_p1)
print("99th-percentile bound:", ae_p99)
print("Normalized minimum:", float(ae_score_norm.min()))
print("Normalized maximum:", float(ae_score_norm.max()))
print("Normalized mean:", float(ae_score_norm.mean()))

# Save the normalized signal used by later stages.
np.save(
    "ae_score_normalized.npy",
    ae_score_norm
)

# Save the two parameters needed for injected-data normalization.
np.savez(
    "ae_normalization_params.npz",
    p1=np.float32(ae_p1),
    p99=np.float32(ae_p99)
)

print("\nSaved:")
print("ae_score_normalized.npy")
print("ae_normalization_params.npz")

In [ ]:
# =========================
# Step 17: Select exact top-2% AE anomalies
# =========================

def select_exact_top_fraction(
    scores,
    fraction=0.02
):
    scores = np.asarray(scores)

    if scores.ndim != 1:
        raise ValueError(
            "Scores must be a one-dimensional array."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Scores contain NaN or infinite values."
        )

    if not 0.0 < fraction < 1.0:
        raise ValueError(
            "Fraction must be between 0 and 1."
        )

    n_samples = scores.shape[0]

    # Preserve your intended top-2% budget.
    k = int(
        np.ceil(
            fraction * n_samples
        )
    )

    top_indices = np.argpartition(
        scores,
        -k
    )[-k:]

    labels = np.zeros(
        n_samples,
        dtype=np.uint8
    )

    labels[top_indices] = 1

    threshold = float(
        np.min(
            scores[top_indices]
        )
    )

    return labels, threshold, top_indices


ae_anomaly_labels, ae_threshold, ae_anomaly_indices = (
    select_exact_top_fraction(
        ae_score_norm,
        fraction=0.02
    )
)

print("AE anomaly threshold:", ae_threshold)
print("Selected AE anomalies:", int(ae_anomaly_labels.sum()))
print(
    "Selected percentage:",
    float(ae_anomaly_labels.mean() * 100)
)

assert ae_anomaly_labels.sum() == int(
    np.ceil(
        0.02 * len(ae_score_norm)
    )
)

# Save only the binary labels.
np.save(
    "ae_anomaly_labels.npy",
    ae_anomaly_labels
)

print("Saved: ae_anomaly_labels.npy")

In [ ]:
# =========================
# Step 2A: Create training subset
# =========================

# Reuse the processed 23-dimensional dataset created earlier.
X_full = X_full_scaled

assert X_full.shape == (X.shape[0], 23)
assert X_full.dtype == np.float32
assert np.isfinite(X_full).all()

print("Using processed dataset:", X_full.shape)

# -------------------------
# Fixed training subset
# -------------------------

sample_size = 300_000
SEED = 42

rng_detector = np.random.default_rng(SEED)

# random_split stores the selected indices in each Subset object.
train_indices = np.asarray(
    train_dataset.indices,
    dtype=np.int64
)

assert train_indices.ndim == 1
assert train_indices.shape[0] == len(train_dataset)

if sample_size > train_indices.shape[0]:
    raise ValueError(
        f"Requested sample size {sample_size} exceeds "
        f"the training-set size {train_indices.shape[0]}."
    )

detector_sample_indices = rng_detector.choice(
    train_indices,
    size=sample_size,
    replace=False
)

X_sample = X_full[detector_sample_indices]

assert X_sample.shape == (sample_size, 23)
assert np.isfinite(X_sample).all()

print("Detector training sample shape:", X_sample.shape)
print("Detector sampling seed:", SEED)

In [ ]:
# =========================
# Step 2B: Train feature-space Isolation Forest
# =========================

from sklearn.ensemble import IsolationForest
import joblib

if_model = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=0.02,
    random_state=SEED,
    n_jobs=-1
)

if_model.fit(X_sample)

joblib.dump(
    if_model,
    "if_feature_model.pkl"
)

print("Feature-space Isolation Forest trained.")
print("Saved: if_feature_model.pkl")

In [ ]:
# =========================
# Step 2C: Compute IF scores for the complete dataset
# =========================

from tqdm import tqdm

def compute_if_scores(
    model,
    X,
    batch_size=100_000
):
    n_samples = X.shape[0]

    scores = np.empty(
        n_samples,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_samples, batch_size),
        desc="IF scoring"
    ):
        end = min(
            start + batch_size,
            n_samples
        )

        batch = X[start:end]

        # IsolationForest decision_function:
        # larger values indicate more normal samples.
        # Negation makes larger values more anomalous.
        batch_scores = -model.decision_function(
            batch
        )

        scores[start:end] = batch_scores.astype(
            np.float32,
            copy=False
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Isolation Forest scores contain NaN or infinite values."
        )

    return scores


IF_scores = compute_if_scores(
    if_model,
    X_full,
    batch_size=100_000
)

print("IF scores computed:", IF_scores.shape)
print("IF score dtype:", IF_scores.dtype)
print("IF score minimum:", float(IF_scores.min()))
print("IF score maximum:", float(IF_scores.max()))

In [ ]:
# =========================
# Step 2D: Normalize feature-space IF scores
# =========================

if_p1 = float(
    np.percentile(
        IF_scores,
        1
    )
)

if_p99 = float(
    np.percentile(
        IF_scores,
        99
    )
)

if if_p99 <= if_p1:
    raise ValueError(
        "Invalid IF normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

IF_scores_clipped = np.clip(
    IF_scores,
    if_p1,
    if_p99
)

IF_feat_norm = (
    IF_scores_clipped - if_p1
) / (
    if_p99 - if_p1
)

IF_feat_norm = IF_feat_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(IF_feat_norm).all()
assert IF_feat_norm.min() >= 0.0
assert IF_feat_norm.max() <= 1.0

print("IF normalized statistics:")
print("1st-percentile bound:", if_p1)
print("99th-percentile bound:", if_p99)
print("Minimum:", float(IF_feat_norm.min()))
print("Maximum:", float(IF_feat_norm.max()))
print("Mean:", float(IF_feat_norm.mean()))

# Save the normalized original-data score.
np.save(
    "if_feature_score_normalized.npy",
    IF_feat_norm
)

# Save the original-data normalization limits.
np.savez(
    "if_feature_normalization_params.npz",
    p1=np.float32(if_p1),
    p99=np.float32(if_p99)
)

print("\nSaved:")
print("if_feature_score_normalized.npy")
print("if_feature_normalization_params.npz")

In [ ]:
# =========================
# Step 3A: Train feature-space LOF
# =========================

from sklearn.neighbors import LocalOutlierFactor
import joblib

# X_sample must already have been generated using SEED = 42.
assert X_sample.shape == (300_000, 23)
assert X_sample.dtype == np.float32
assert np.isfinite(X_sample).all()

lof_model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
    n_jobs=-1
)

lof_model.fit(X_sample)

joblib.dump(
    lof_model,
    "lof_feature_model.pkl"
)

print("Feature-space LOF trained.")
print("Training sample shape:", X_sample.shape)
print("Seed used for sample selection:", SEED)
print("Saved: lof_feature_model.pkl")

In [ ]:
# =========================
# Step 3B: Compute feature-space LOF scores
# =========================

from tqdm import tqdm

def compute_lof_scores(
    model,
    X,
    batch_size=100_000
):
    n_samples = X.shape[0]

    scores = np.empty(
        n_samples,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_samples, batch_size),
        desc="LOF scoring"
    ):
        end = min(
            start + batch_size,
            n_samples
        )

        batch = X[start:end]

        # score_samples:
        # larger values indicate more normal samples.
        # Negation makes larger values indicate stronger anomalies.
        batch_scores = -model.score_samples(
            batch
        )

        scores[start:end] = batch_scores.astype(
            np.float32,
            copy=False
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "LOF scores contain NaN or infinite values."
        )

    return scores


LOF_scores = compute_lof_scores(
    lof_model,
    X_full,
    batch_size=100_000
)

print("LOF scores computed:", LOF_scores.shape)
print("LOF score dtype:", LOF_scores.dtype)
print("LOF score minimum:", float(LOF_scores.min()))
print("LOF score maximum:", float(LOF_scores.max()))
print("LOF score mean:", float(LOF_scores.mean()))

In [ ]:
# =========================
# Step 3C: Normalize feature-space LOF scores
# =========================

lof_p1 = float(
    np.percentile(
        LOF_scores,
        1
    )
)

lof_p99 = float(
    np.percentile(
        LOF_scores,
        99
    )
)

if lof_p99 <= lof_p1:
    raise ValueError(
        "Invalid LOF normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

LOF_scores_clipped = np.clip(
    LOF_scores,
    lof_p1,
    lof_p99
)

LOF_feat_norm = (
    LOF_scores_clipped - lof_p1
) / (
    lof_p99 - lof_p1
)

LOF_feat_norm = LOF_feat_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(LOF_feat_norm).all()
assert LOF_feat_norm.min() >= 0.0
assert LOF_feat_norm.max() <= 1.0

print("LOF normalized statistics:")
print("1st-percentile bound:", lof_p1)
print("99th-percentile bound:", lof_p99)
print("Minimum:", float(LOF_feat_norm.min()))
print("Maximum:", float(LOF_feat_norm.max()))
print("Mean:", float(LOF_feat_norm.mean()))

# Save the normalized feature-space LOF signal.
np.save(
    "lof_feature_score_normalized.npy",
    LOF_feat_norm
)

# Save the original-data normalization parameters.
np.savez(
    "lof_feature_normalization_params.npz",
    p1=np.float32(lof_p1),
    p99=np.float32(lof_p99)
)

print("\nSaved:")
print("lof_feature_score_normalized.npy")
print("lof_feature_normalization_params.npz")

In [ ]:
# =========================
# Step 4A: Fit feature-space Mahalanobis model
# =========================

# X_sample was selected previously using SEED = 42.
assert X_sample.shape == (300_000, 23)
assert X_sample.dtype == np.float32
assert np.isfinite(X_sample).all()

# Use float64 for covariance estimation and inversion.
X_sample_md = X_sample.astype(
    np.float64,
    copy=False
)

mu_md = np.mean(
    X_sample_md,
    axis=0
)

cov_md = np.cov(
    X_sample_md,
    rowvar=False
)

# Regularize the covariance matrix for numerical stability.
regularization = 1e-6

cov_md = cov_md + (
    np.eye(
        cov_md.shape[0],
        dtype=np.float64
    )
    * regularization
)

# Pseudo-inverse is safer when features are correlated.
inv_cov_md = np.linalg.pinv(
    cov_md
)

assert mu_md.shape == (23,)
assert cov_md.shape == (23, 23)
assert inv_cov_md.shape == (23, 23)
assert np.isfinite(mu_md).all()
assert np.isfinite(cov_md).all()
assert np.isfinite(inv_cov_md).all()

print("Feature-space Mahalanobis parameters computed.")
print("Training sample shape:", X_sample.shape)
print("Seed used for sample selection:", SEED)
print("Regularization:", regularization)

In [ ]:
# =========================
# Step 4B: Compute feature-space Mahalanobis scores
# =========================

from tqdm import tqdm

def compute_mahalanobis_scores(
    X,
    mean_vector,
    inverse_covariance,
    batch_size=100_000
):
    n_samples = X.shape[0]

    scores = np.empty(
        n_samples,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_samples, batch_size),
        desc="MD scoring"
    ):
        end = min(
            start + batch_size,
            n_samples
        )

        # Use float64 during the matrix calculation.
        batch = X[start:end].astype(
            np.float64,
            copy=False
        )

        diff = batch - mean_vector

        squared_distance = np.einsum(
            "ij,jk,ik->i",
            diff,
            inverse_covariance,
            diff,
            optimize=True
        )

        # Protect against very small negative values caused by
        # floating-point round-off.
        squared_distance = np.maximum(
            squared_distance,
            0.0
        )

        scores[start:end] = np.sqrt(
            squared_distance
        ).astype(
            np.float32,
            copy=False
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Mahalanobis scores contain NaN or infinite values."
        )

    return scores


MD_raw = compute_mahalanobis_scores(
    X=X_full,
    mean_vector=mu_md,
    inverse_covariance=inv_cov_md,
    batch_size=100_000
)

print("Mahalanobis scores computed:", MD_raw.shape)
print("MD score dtype:", MD_raw.dtype)
print("MD score minimum:", float(MD_raw.min()))
print("MD score maximum:", float(MD_raw.max()))
print("MD score mean:", float(MD_raw.mean()))

In [ ]:
# =========================
# Step 4C: Normalize feature-space Mahalanobis scores
# =========================

md_p1 = float(
    np.percentile(
        MD_raw,
        1
    )
)

md_p99 = float(
    np.percentile(
        MD_raw,
        99
    )
)

if md_p99 <= md_p1:
    raise ValueError(
        "Invalid MD normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

MD_clipped = np.clip(
    MD_raw,
    md_p1,
    md_p99
)

MD_feat_norm = (
    MD_clipped - md_p1
) / (
    md_p99 - md_p1
)

MD_feat_norm = MD_feat_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(MD_feat_norm).all()
assert MD_feat_norm.min() >= 0.0
assert MD_feat_norm.max() <= 1.0

print("MD normalized statistics:")
print("1st-percentile bound:", md_p1)
print("99th-percentile bound:", md_p99)
print("Minimum:", float(MD_feat_norm.min()))
print("Maximum:", float(MD_feat_norm.max()))
print("Mean:", float(MD_feat_norm.mean()))

np.save(
    "md_feature_score_normalized.npy",
    MD_feat_norm
)

np.savez(
    "md_feature_normalization_params.npz",
    p1=np.float32(md_p1),
    p99=np.float32(md_p99)
)

print("\nSaved:")
print("md_feature_score_normalized.npy")
print("md_feature_normalization_params.npz")

In [ ]:
# =========================
# Step 4D: Save Mahalanobis model parameters
# =========================

np.savez(
    "md_feature_model_params.npz",
    mean=mu_md.astype(np.float64),
    inverse_covariance=inv_cov_md.astype(np.float64),
    regularization=np.float64(regularization)
)

print("Saved: md_feature_model_params.npz")

In [ ]:
# =========================
# Step E1-E2: Train Isolation Forest on embeddings
# =========================

from sklearn.ensemble import IsolationForest
import joblib

# Reuse the exact detector sample indices selected earlier with SEED.
E_sample = embeddings[detector_sample_indices].astype(
    np.float32,
    copy=False
)

assert E_sample.shape == (300_000, 16)
assert np.isfinite(E_sample).all()

if_emb_model = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=0.02,
    random_state=SEED,
    n_jobs=-1
)

if_emb_model.fit(E_sample)

joblib.dump(
    if_emb_model,
    "if_embedding_model.pkl"
)

print("Embedding-space Isolation Forest trained.")
print("Training sample shape:", E_sample.shape)
print("Seed:", SEED)
print("Saved: if_embedding_model.pkl")

In [ ]:
# =========================
# Step E3: Compute IF scores in embedding space
# =========================

IF_emb_scores = compute_if_scores(
    model=if_emb_model,
    X=embeddings,
    batch_size=100_000
)

print("Embedding-space IF scores computed:", IF_emb_scores.shape)
print("Score dtype:", IF_emb_scores.dtype)
print("Minimum:", float(IF_emb_scores.min()))
print("Maximum:", float(IF_emb_scores.max()))
print("Mean:", float(IF_emb_scores.mean()))

In [ ]:
# =========================
# Step E4: Normalize embedding-space IF scores
# =========================

if_emb_p1 = float(
    np.percentile(
        IF_emb_scores,
        1
    )
)

if_emb_p99 = float(
    np.percentile(
        IF_emb_scores,
        99
    )
)

if if_emb_p99 <= if_emb_p1:
    raise ValueError(
        "Invalid embedding-space IF normalization range."
    )

IF_emb_scores_clipped = np.clip(
    IF_emb_scores,
    if_emb_p1,
    if_emb_p99
)

IF_emb_norm = (
    IF_emb_scores_clipped - if_emb_p1
) / (
    if_emb_p99 - if_emb_p1
)

IF_emb_norm = IF_emb_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(IF_emb_norm).all()
assert IF_emb_norm.min() >= 0.0
assert IF_emb_norm.max() <= 1.0

print("Embedding-space IF normalized statistics:")
print("1st-percentile bound:", if_emb_p1)
print("99th-percentile bound:", if_emb_p99)
print("Minimum:", float(IF_emb_norm.min()))
print("Maximum:", float(IF_emb_norm.max()))
print("Mean:", float(IF_emb_norm.mean()))

np.save(
    "if_embedding_score_normalized.npy",
    IF_emb_norm
)

np.savez(
    "if_embedding_normalization_params.npz",
    p1=np.float32(if_emb_p1),
    p99=np.float32(if_emb_p99)
)

print("\nSaved:")
print("if_embedding_score_normalized.npy")
print("if_embedding_normalization_params.npz")

In [ ]:
# =========================
# Step E5: Train LOF on embeddings
# =========================

from sklearn.neighbors import LocalOutlierFactor
import joblib

assert E_sample.shape == (300_000, 16)
assert E_sample.dtype == np.float32
assert np.isfinite(E_sample).all()

lof_emb_model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
    n_jobs=-1
)

lof_emb_model.fit(E_sample)

joblib.dump(
    lof_emb_model,
    "lof_embedding_model.pkl"
)

print("Embedding-space LOF trained.")
print("Training sample shape:", E_sample.shape)
print("Seed used for sample selection:", SEED)
print("Saved: lof_embedding_model.pkl")

In [ ]:
# =========================
# Step E6: Compute LOF scores in embedding space
# =========================

LOF_emb_scores = compute_lof_scores(
    model=lof_emb_model,
    X=embeddings,
    batch_size=100_000
)

print("Embedding-space LOF scores computed:", LOF_emb_scores.shape)
print("Score dtype:", LOF_emb_scores.dtype)
print("Minimum:", float(LOF_emb_scores.min()))
print("Maximum:", float(LOF_emb_scores.max()))
print("Mean:", float(LOF_emb_scores.mean()))

In [ ]:
# =========================
# Step E7: Normalize embedding-space LOF scores
# =========================

lof_emb_p1 = float(
    np.percentile(
        LOF_emb_scores,
        1
    )
)

lof_emb_p99 = float(
    np.percentile(
        LOF_emb_scores,
        99
    )
)

if lof_emb_p99 <= lof_emb_p1:
    raise ValueError(
        "Invalid embedding-space LOF normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

LOF_emb_scores_clipped = np.clip(
    LOF_emb_scores,
    lof_emb_p1,
    lof_emb_p99
)

LOF_emb_norm = (
    LOF_emb_scores_clipped - lof_emb_p1
) / (
    lof_emb_p99 - lof_emb_p1
)

LOF_emb_norm = LOF_emb_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(LOF_emb_norm).all()
assert LOF_emb_norm.min() >= 0.0
assert LOF_emb_norm.max() <= 1.0

print("Embedding-space LOF normalized statistics:")
print("1st-percentile bound:", lof_emb_p1)
print("99th-percentile bound:", lof_emb_p99)
print("Minimum:", float(LOF_emb_norm.min()))
print("Maximum:", float(LOF_emb_norm.max()))
print("Mean:", float(LOF_emb_norm.mean()))

np.save(
    "lof_embedding_score_normalized.npy",
    LOF_emb_norm
)

np.savez(
    "lof_embedding_normalization_params.npz",
    p1=np.float32(lof_emb_p1),
    p99=np.float32(lof_emb_p99)
)

print("\nSaved:")
print("lof_embedding_score_normalized.npy")
print("lof_embedding_normalization_params.npz")

In [ ]:
# =========================
# Step E8: Fit Mahalanobis model on embeddings
# =========================

assert E_sample.shape == (300_000, 16)
assert E_sample.dtype == np.float32
assert np.isfinite(E_sample).all()

# Use float64 for covariance estimation and inversion.
E_sample_md = E_sample.astype(
    np.float64,
    copy=False
)

mu_emb = np.mean(
    E_sample_md,
    axis=0
)

cov_emb = np.cov(
    E_sample_md,
    rowvar=False
)

md_emb_regularization = 1e-6

cov_emb = cov_emb + (
    np.eye(
        cov_emb.shape[0],
        dtype=np.float64
    )
    * md_emb_regularization
)

inv_cov_emb = np.linalg.pinv(
    cov_emb
)

assert mu_emb.shape == (16,)
assert cov_emb.shape == (16, 16)
assert inv_cov_emb.shape == (16, 16)

assert np.isfinite(mu_emb).all()
assert np.isfinite(cov_emb).all()
assert np.isfinite(inv_cov_emb).all()

print("Embedding-space Mahalanobis parameters computed.")
print("Training sample shape:", E_sample.shape)
print("Seed used for sample selection:", SEED)
print("Regularization:", md_emb_regularization)

In [ ]:
np.savez(
    "md_embedding_model_params.npz",
    mean=mu_emb.astype(np.float64),
    inverse_covariance=inv_cov_emb.astype(np.float64),
    regularization=np.float64(md_emb_regularization)
)

print("Saved: md_embedding_model_params.npz")

In [ ]:
# =========================
# Step E9: Compute Mahalanobis scores in embedding space
# =========================

MD_emb_raw = compute_mahalanobis_scores(
    X=embeddings,
    mean_vector=mu_emb,
    inverse_covariance=inv_cov_emb,
    batch_size=100_000
)

print("Embedding-space MD scores computed:", MD_emb_raw.shape)
print("Score dtype:", MD_emb_raw.dtype)
print("Minimum:", float(MD_emb_raw.min()))
print("Maximum:", float(MD_emb_raw.max()))
print("Mean:", float(MD_emb_raw.mean()))

In [ ]:
# =========================
# Step E10: Normalize embedding-space Mahalanobis scores
# =========================

md_emb_p1 = float(
    np.percentile(
        MD_emb_raw,
        1
    )
)

md_emb_p99 = float(
    np.percentile(
        MD_emb_raw,
        99
    )
)

if md_emb_p99 <= md_emb_p1:
    raise ValueError(
        "Invalid embedding-space MD normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

MD_emb_clipped = np.clip(
    MD_emb_raw,
    md_emb_p1,
    md_emb_p99
)

MD_emb_norm = (
    MD_emb_clipped - md_emb_p1
) / (
    md_emb_p99 - md_emb_p1
)

MD_emb_norm = MD_emb_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(MD_emb_norm).all()
assert MD_emb_norm.min() >= 0.0
assert MD_emb_norm.max() <= 1.0

print("Embedding-space MD normalized statistics:")
print("1st-percentile bound:", md_emb_p1)
print("99th-percentile bound:", md_emb_p99)
print("Minimum:", float(MD_emb_norm.min()))
print("Maximum:", float(MD_emb_norm.max()))
print("Mean:", float(MD_emb_norm.mean()))

np.save(
    "md_embedding_score_normalized.npy",
    MD_emb_norm
)

np.savez(
    "md_embedding_normalization_params.npz",
    p1=np.float32(md_emb_p1),
    p99=np.float32(md_emb_p99)
)

print("\nSaved:")
print("md_embedding_score_normalized.npy")
print("md_embedding_normalization_params.npz")

In [ ]:
# =========================
# Convert normalized scores to exact top-2% anomaly labels
# =========================

# The function select_exact_top_fraction() was defined earlier.
# It returns:
#   labels, threshold, selected_indices

score_sets = {
    "AE": ae_score_norm,
    "IF feature": IF_feat_norm,
    "IF embedding": IF_emb_norm,
    "LOF feature": LOF_feat_norm,
    "LOF embedding": LOF_emb_norm,
    "MD feature": MD_feat_norm,
    "MD embedding": MD_emb_norm,
}

# Validate all score arrays before thresholding.
n_samples = X.shape[0]

for name, scores in score_sets.items():
    assert scores.shape == (n_samples,), (
        f"{name} score shape mismatch: "
        f"expected {(n_samples,)}, found {scores.shape}."
    )

    assert np.isfinite(scores).all(), (
        f"{name} scores contain NaN or infinite values."
    )

# AE
AE_labels, AE_thr, _ = select_exact_top_fraction(
    ae_score_norm,
    fraction=0.02
)

# IF
IF_feat_labels, IF_feat_thr, _ = select_exact_top_fraction(
    IF_feat_norm,
    fraction=0.02
)

IF_emb_labels, IF_emb_thr, _ = select_exact_top_fraction(
    IF_emb_norm,
    fraction=0.02
)

# LOF
LOF_feat_labels, LOF_feat_thr, _ = select_exact_top_fraction(
    LOF_feat_norm,
    fraction=0.02
)

LOF_emb_labels, LOF_emb_thr, _ = select_exact_top_fraction(
    LOF_emb_norm,
    fraction=0.02
)

# MD
MD_feat_labels, MD_feat_thr, _ = select_exact_top_fraction(
    MD_feat_norm,
    fraction=0.02
)

MD_emb_labels, MD_emb_thr, _ = select_exact_top_fraction(
    MD_emb_norm,
    fraction=0.02
)

# -------------------------
# Print thresholds
# -------------------------

print("Thresholds:")
print("AE:", AE_thr)
print("IF feature:", IF_feat_thr)
print("IF embedding:", IF_emb_thr)
print("LOF feature:", LOF_feat_thr)
print("LOF embedding:", LOF_emb_thr)
print("MD feature:", MD_feat_thr)
print("MD embedding:", MD_emb_thr)

# -------------------------
# Print anomaly counts
# -------------------------

label_sets = {
    "AE": AE_labels,
    "IF feature": IF_feat_labels,
    "IF embedding": IF_emb_labels,
    "LOF feature": LOF_feat_labels,
    "LOF embedding": LOF_emb_labels,
    "MD feature": MD_feat_labels,
    "MD embedding": MD_emb_labels,
}

expected_anomalies = int(
    np.ceil(
        0.02 * n_samples
    )
)

print("\nExpected anomaly count:", expected_anomalies)
print("\nAnomaly counts:")

for name, labels in label_sets.items():
    anomaly_count = int(labels.sum())
    anomaly_percentage = float(labels.mean() * 100)

    print(
        f"{name}: {anomaly_count} "
        f"({anomaly_percentage:.6f}%)"
    )

    assert anomaly_count == expected_anomalies, (
        f"{name} did not select exactly "
        f"{expected_anomalies} anomalies."
    )

print("\nAll score sets use the same exact top-2% anomaly budget.")

In [ ]:
# =========================
# Save detector anomaly labels
# =========================

np.savez_compressed(
    "detector_anomaly_labels.npz",
    ae=AE_labels,
    if_feature=IF_feat_labels,
    if_embedding=IF_emb_labels,
    lof_feature=LOF_feat_labels,
    lof_embedding=LOF_emb_labels,
    md_feature=MD_feat_labels,
    md_embedding=MD_emb_labels
)

print("Saved: detector_anomaly_labels.npz")

In [ ]:
print("IF overlap:", np.sum(IF_feat_labels & IF_emb_labels))
print("LOF overlap:", np.sum(LOF_feat_labels & LOF_emb_labels))
print("MD overlap:", np.sum(MD_feat_labels & MD_emb_labels))

In [ ]:
# =========================
# Feature-space and embedding-space detector overlap
# =========================

def overlap_statistics(
    feature_labels,
    embedding_labels
):
    feature_mask = feature_labels.astype(bool)
    embedding_mask = embedding_labels.astype(bool)

    intersection = int(
        np.logical_and(
            feature_mask,
            embedding_mask
        ).sum()
    )

    union = int(
        np.logical_or(
            feature_mask,
            embedding_mask
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else 0.0
    )

    overlap_fraction = (
        intersection / feature_mask.sum()
        if feature_mask.sum() > 0
        else 0.0
    )

    return intersection, union, jaccard, overlap_fraction


detector_pairs = {
    "IF": (
        IF_feat_labels,
        IF_emb_labels
    ),
    "LOF": (
        LOF_feat_labels,
        LOF_emb_labels
    ),
    "MD": (
        MD_feat_labels,
        MD_emb_labels
    ),
}

for detector_name, (
    feature_labels,
    embedding_labels
) in detector_pairs.items():

    intersection, union, jaccard, overlap_fraction = (
        overlap_statistics(
            feature_labels,
            embedding_labels
        )
    )

    print(f"\n{detector_name}")
    print("Intersection:", intersection)
    print("Union:", union)
    print("Jaccard similarity:", f"{jaccard:.6f}")
    print(
        "Shared fraction of each top-2% set:",
        f"{overlap_fraction * 100:.2f}%"
    )

In [ ]:
# =========================
# Create the base anomaly score
# =========================

base_score_components = np.stack(
    [
        ae_score_norm,
        IF_feat_norm,
        LOF_feat_norm,
        MD_feat_norm,
        IF_emb_norm,
        LOF_emb_norm,
        MD_emb_norm,
    ],
    axis=0
)

assert base_score_components.shape == (
    7,
    X.shape[0]
)

assert np.isfinite(
    base_score_components
).all()

A_base = np.mean(
    base_score_components,
    axis=0,
    dtype=np.float32
)

A_base = A_base.astype(
    np.float32,
    copy=False
)

assert A_base.shape == (X.shape[0],)
assert np.isfinite(A_base).all()

print("Base anomaly score ready:", A_base.shape)
print("Minimum:", float(A_base.min()))
print("Maximum:", float(A_base.max()))
print("Mean:", float(A_base.mean()))
print("Median:", float(np.median(A_base)))

In [ ]:
# =========================
# Load and validate the spatial mask
# =========================

valid_mask = np.load(
    "valid_mask_1.npy"
)

assert valid_mask.ndim == 2, (
    "valid_mask must be a two-dimensional raster mask."
)

# Convert to Boolean if stored as 0/1 integers.
valid_mask = valid_mask.astype(
    bool,
    copy=False
)

mask_pixel_count = int(
    valid_mask.sum()
)

dataset_row_count = int(
    X.shape[0]
)

print("Valid-mask shape:", valid_mask.shape)
print("Mask pixels:", mask_pixel_count)
print("Dataset rows:", dataset_row_count)

assert mask_pixel_count == dataset_row_count, (
    "The number of valid raster pixels does not match "
    "the number of rows in X."
)

print("\nThe valid mask is consistent with the feature matrix.")

In [ ]:
# Stage: Spatial Consistency / Spatial Reasoning Stage
# Spatial anomaly score S(x)

In [ ]:
# =========================
# Step 1: Reconstruct base anomaly score as raster
# =========================

A_base_raster = np.full(
    valid_mask.shape,
    np.nan,
    dtype=np.float32
)

A_base_raster[valid_mask] = A_base

valid_score_count = int(
    np.isfinite(A_base_raster).sum()
)

print("A_base raster shape:", A_base_raster.shape)
print("Valid anomaly-score pixels:", valid_score_count)

assert valid_score_count == X.shape[0], (
    "The number of reconstructed raster scores does not match "
    "the number of dataset rows."
)

assert np.isfinite(
    A_base_raster[valid_mask]
).all()

print("Base anomaly score reconstructed successfully.")

In [ ]:
# =========================
# Step 2: Compute local spatial inconsistency
# =========================

from scipy.ndimage import uniform_filter

def nan_local_mean(
    arr,
    size=5
):
    """
    Compute a neighborhood mean while excluding NaN cells.
    """
    valid = np.isfinite(arr)

    filled = np.where(
        valid,
        arr,
        0.0
    ).astype(np.float32)

    kernel_area = float(size * size)

    local_sum = uniform_filter(
        filled,
        size=size,
        mode="nearest"
    ) * kernel_area

    local_count = uniform_filter(
        valid.astype(np.float32),
        size=size,
        mode="nearest"
    ) * kernel_area

    local_mean = local_sum / np.maximum(
        local_count,
        1.0
    )

    return local_mean, local_count


window_size = 5

# -------------------------
# Local mean
# -------------------------

local_mean, local_count = nan_local_mean(
    A_base_raster,
    size=window_size
)

# -------------------------
# Local second moment
# -------------------------

A_base_squared = np.where(
    np.isfinite(A_base_raster),
    A_base_raster ** 2,
    np.nan
)

local_second_moment, _ = nan_local_mean(
    A_base_squared,
    size=window_size
)

# -------------------------
# Local variance and std
# -------------------------

local_variance = np.maximum(
    local_second_moment - local_mean ** 2,
    0.0
)

local_std = np.sqrt(
    local_variance
)

# -------------------------
# Local z-score magnitude
# -------------------------

S_raster = (
    np.abs(
        A_base_raster - local_mean
    )
    /
    (
        local_std + 1e-6
    )
)

# Retain valid pixels in the same order as X.
S = S_raster[valid_mask].astype(
    np.float32,
    copy=False
)

assert S.shape == (X.shape[0],)
assert np.isfinite(S).all()

print("Spatial score shape:", S.shape)
print("Minimum:", float(S.min()))
print("Maximum:", float(S.max()))
print("Mean:", float(S.mean()))
print("Median:", float(np.median(S)))

In [ ]:
# =========================
# Step 3: Normalize spatial inconsistency score
# =========================

spatial_p1 = float(
    np.percentile(
        S,
        1
    )
)

spatial_p99 = float(
    np.percentile(
        S,
        99
    )
)

if spatial_p99 <= spatial_p1:
    raise ValueError(
        "Invalid spatial-score normalization range."
    )

S_clipped = np.clip(
    S,
    spatial_p1,
    spatial_p99
)

S_norm = (
    S_clipped - spatial_p1
) / (
    spatial_p99 - spatial_p1
)

S_norm = S_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(S_norm).all()
assert S_norm.min() >= 0.0
assert S_norm.max() <= 1.0

print("Normalized spatial-score statistics:")
print("1st-percentile bound:", spatial_p1)
print("99th-percentile bound:", spatial_p99)
print("Minimum:", float(S_norm.min()))
print("Maximum:", float(S_norm.max()))
print("Mean:", float(S_norm.mean()))
print("95th percentile:", float(np.percentile(S_norm, 95)))
print("98th percentile:", float(np.percentile(S_norm, 98)))

# Save the signal used in later stages.
np.save(
    "spatial_score_normalized.npy",
    S_norm
)

# Save original-data normalization limits for injected evaluation.
np.savez(
    "spatial_score_normalization_params.npz",
    p1=np.float32(spatial_p1),
    p99=np.float32(spatial_p99),
    window_size=np.int32(window_size)
)

print("\nSaved:")
print("spatial_score_normalized.npy")
print("spatial_score_normalization_params.npz")

In [ ]:
# =========================
# Step 4: Create spatially refined Full score
# =========================

full_base_weight = 0.75
full_spatial_weight = 0.25

A_spatial = (
    full_base_weight * A_base
    +
    full_spatial_weight * S_norm
).astype(
    np.float32,
    copy=False
)

assert A_spatial.shape == (X.shape[0],)
assert np.isfinite(A_spatial).all()

print("Spatially refined score shape:", A_spatial.shape)
print("Minimum:", float(A_spatial.min()))
print("Maximum:", float(A_spatial.max()))
print("Mean:", float(A_spatial.mean()))
print("95th percentile:", float(np.percentile(A_spatial, 95)))
print("98th percentile:", float(np.percentile(A_spatial, 98)))
print("99th percentile:", float(np.percentile(A_spatial, 99)))

In [ ]:
# =========================
# Step 5: Normalize spatially refined score
# =========================

a_spatial_p1 = float(
    np.percentile(
        A_spatial,
        1
    )
)

a_spatial_p99 = float(
    np.percentile(
        A_spatial,
        99
    )
)

if a_spatial_p99 <= a_spatial_p1:
    raise ValueError(
        "Invalid A_spatial normalization range."
    )

A_spatial_clipped = np.clip(
    A_spatial,
    a_spatial_p1,
    a_spatial_p99
)

A_spatial_norm = (
    A_spatial_clipped - a_spatial_p1
) / (
    a_spatial_p99 - a_spatial_p1
)

A_spatial_norm = A_spatial_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(A_spatial_norm).all()
assert A_spatial_norm.min() >= 0.0
assert A_spatial_norm.max() <= 1.0

print("A_spatial normalization completed.")
print("1st-percentile bound:", a_spatial_p1)
print("99th-percentile bound:", a_spatial_p99)
print("Minimum:", float(A_spatial_norm.min()))
print("Maximum:", float(A_spatial_norm.max()))
print("Mean:", float(A_spatial_norm.mean()))

In [ ]:
np.savez(
    "full_spatial_normalization_params.npz",
    p1=np.float32(a_spatial_p1),
    p99=np.float32(a_spatial_p99),
    base_weight=np.float32(full_base_weight),
    spatial_weight=np.float32(full_spatial_weight)
)

print("Saved: full_spatial_normalization_params.npz")

In [ ]:
# =========================
# Step 6: Spatial-score percentile summary
# =========================

percentile_levels = [
    50,
    75,
    90,
    95,
    98,
    99
]

spatial_percentiles = np.percentile(
    S_norm,
    percentile_levels
)

for percentile, value in zip(
    percentile_levels,
    spatial_percentiles
):
    print(
        f"{percentile}th percentile: "
        f"{value:.4f}"
    )

In [ ]:
# Stage 6: Domain Context Score (C(x))
# So far we have: A_base → anomaly in feature space, S(x) → anomaly in spatial neighborhood. Now we add: C(x): anomaly plausibility given domain.
# We will use: Unsupervised context model (MLP Autoencoder).

In [ ]:
# =========================
# Step 1: Construct context feature matrix
# =========================

X_context = np.concatenate(
    [
        X_landuse_scaled,          # 7: land-use indicators
        X_surface_scaled,          # 3: NDVI, NDWI, NDBI
        X_infra_scaled,            # 1: road distance
        X_hydro_scaled,            # 3: flow_acc, flow_contrast, river_dist
        X_terrain_scaled[:, 0:2],  # 2: elevation and slope
    ],
    axis=1
).astype(
    np.float32,
    copy=False
)

expected_context_shape = (
    X.shape[0],
    16
)

assert X_context.shape == expected_context_shape, (
    f"Expected context shape {expected_context_shape}, "
    f"but found {X_context.shape}."
)

assert np.isfinite(X_context).all(), (
    "X_context contains NaN or infinite values."
)

print("Context feature shape:", X_context.shape)
print("Context dtype:", X_context.dtype)

In [ ]:
# =========================
# Step 2: Create context training subset
# =========================

assert detector_sample_indices.shape == (300_000,)
assert detector_sample_indices.min() >= 0
assert detector_sample_indices.max() < X_context.shape[0]

X_context_sample = X_context[
    detector_sample_indices
].astype(
    np.float32,
    copy=False
)

assert X_context_sample.shape == (
    300_000,
    16
)

assert np.isfinite(X_context_sample).all()

print("Context training sample:", X_context_sample.shape)
print("Seed used for sample selection:", SEED)

In [ ]:
# =========================
# Step 3: Context Dataset
# =========================

from torch.utils.data import Dataset, DataLoader

class ContextDataset(Dataset):
    def __init__(self, X):
        if X.ndim != 2:
            raise ValueError(
                "Context input must be a two-dimensional array."
            )

        if X.shape[1] != 16:
            raise ValueError(
                f"Expected 16 context features, found {X.shape[1]}."
            )

        if X.dtype != np.float32:
            X = X.astype(
                np.float32,
                copy=False
            )

        if not np.isfinite(X).all():
            raise ValueError(
                "Context input contains NaN or infinite values."
            )

        self.X = X

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(
            self.X[idx]
        )

In [ ]:
# =========================
# Step 4: Context training DataLoader
# =========================

context_batch_size = 2048

context_dataset = ContextDataset(
    X_context_sample
)

context_generator = torch.Generator().manual_seed(
    SEED
)

context_loader = DataLoader(
    context_dataset,
    batch_size=context_batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
    generator=context_generator
)

print("Context training samples:", len(context_dataset))
print("Context training batches:", len(context_loader))
print("Context batch size:", context_batch_size)

In [ ]:
# =========================
# Step 5: Context autoencoder
# =========================

class ContextAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

    def forward(self, x):
        embedding = self.encoder(x)
        reconstruction = self.decoder(embedding)

        return reconstruction, embedding

In [ ]:
# =========================
# Step 6: Train context autoencoder
# =========================

model_ctx = ContextAutoencoder().to(
    device
)

context_optimizer = torch.optim.Adam(
    model_ctx.parameters(),
    lr=1e-3
)

context_criterion = nn.MSELoss(
    reduction="mean"
)

context_epochs = 10
context_model_path = "context_autoencoder_best.pth"

context_train_losses = []

best_context_loss = float("inf")
best_context_epoch = 0

for epoch in range(context_epochs):
    model_ctx.train()

    total_loss = 0.0
    total_samples = 0

    for batch in context_loader:
        batch = batch.to(
            device,
            non_blocking=True
        )

        context_optimizer.zero_grad(
            set_to_none=True
        )

        reconstruction, _ = model_ctx(
            batch
        )

        loss = context_criterion(
            reconstruction,
            batch
        )

        loss.backward()
        context_optimizer.step()

        current_batch_size = batch.size(0)

        total_loss += (
            loss.item()
            * current_batch_size
        )

        total_samples += current_batch_size

    epoch_loss = total_loss / total_samples
    context_train_losses.append(epoch_loss)

    print(
        f"Epoch {epoch + 1}/{context_epochs}: "
        f"Loss = {epoch_loss:.6f}"
    )

    if epoch_loss < best_context_loss:
        best_context_loss = epoch_loss
        best_context_epoch = epoch + 1

        torch.save(
            model_ctx.state_dict(),
            context_model_path
        )

        print("Best context checkpoint saved.")

# Reload the best checkpoint.
model_ctx.load_state_dict(
    torch.load(
        context_model_path,
        map_location=device
    )
)

model_ctx.eval()

print("\nContext autoencoder training completed.")
print("Best epoch:", best_context_epoch)
print(
    "Best training loss:",
    f"{best_context_loss:.6f}"
)
print(
    "Saved:",
    context_model_path
)

In [ ]:
# =========================
# Step 7: Full context DataLoader
# =========================

full_context_dataset = ContextDataset(
    X_context
)

full_context_loader = DataLoader(
    full_context_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

print("Full context dataset:", len(full_context_dataset))
print("Full context batches:", len(full_context_loader))

In [ ]:
# =========================
# Step 8: Compute context anomaly scores
# =========================

model_ctx.eval()

n_samples = X_context.shape[0]

C_scores = np.empty(
    n_samples,
    dtype=np.float32
)

start_idx = 0

with torch.no_grad():
    for batch in tqdm(
        full_context_loader,
        desc="Context scoring"
    ):
        batch = batch.to(
            device,
            non_blocking=True
        )

        reconstruction, _ = model_ctx(
            batch
        )

        error = torch.mean(
            (reconstruction - batch) ** 2,
            dim=1
        )

        error_np = error.cpu().numpy().astype(
            np.float32,
            copy=False
        )

        current_batch_size = error_np.shape[0]
        end_idx = start_idx + current_batch_size

        C_scores[start_idx:end_idx] = error_np

        start_idx = end_idx

assert start_idx == n_samples
assert C_scores.shape == (n_samples,)
assert np.isfinite(C_scores).all()

print("Context-score shape:", C_scores.shape)
print("Minimum:", float(C_scores.min()))
print("Maximum:", float(C_scores.max()))
print("Mean:", float(C_scores.mean()))
print("Median:", float(np.median(C_scores)))

In [ ]:
# =========================
# Step 9: Normalize context anomaly score
# =========================

context_p1 = float(
    np.percentile(
        C_scores,
        1
    )
)

context_p99 = float(
    np.percentile(
        C_scores,
        99
    )
)

if context_p99 <= context_p1:
    raise ValueError(
        "Invalid context-score normalization range: "
        "the 99th percentile must exceed the 1st percentile."
    )

C_scores_clipped = np.clip(
    C_scores,
    context_p1,
    context_p99
)

C_norm = (
    C_scores_clipped - context_p1
) / (
    context_p99 - context_p1
)

C_norm = C_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(C_norm).all()
assert C_norm.min() >= 0.0
assert C_norm.max() <= 1.0

print("Normalized context-score statistics:")
print("1st-percentile bound:", context_p1)
print("99th-percentile bound:", context_p99)
print("Minimum:", float(C_norm.min()))
print("Maximum:", float(C_norm.max()))
print("Mean:", float(C_norm.mean()))
print("95th percentile:", float(np.percentile(C_norm, 95)))
print("98th percentile:", float(np.percentile(C_norm, 98)))

np.save(
    "context_score_normalized.npy",
    C_norm
)

np.savez(
    "context_score_normalization_params.npz",
    p1=np.float32(context_p1),
    p99=np.float32(context_p99)
)

print("\nSaved:")
print("context_score_normalized.npy")
print("context_score_normalization_params.npz")

In [ ]:
# Stage 7: Final Fusion + Reliability Filtering
# This is our final model definition

In [ ]:
# ===============================
# Stage 7A: Full multi-signal reliability
# ===============================

# All arrays must represent the same pixels in the same order.
full_score_components = {
    "IF feature": IF_feat_norm,
    "LOF feature": LOF_feat_norm,
    "MD feature": MD_feat_norm,
    "IF embedding": IF_emb_norm,
    "LOF embedding": LOF_emb_norm,
    "MD embedding": MD_emb_norm,
    "AE reconstruction": ae_score_norm,
    "Spatial": S_norm,
    "Context": C_norm,
}

n_samples = X.shape[0]

for name, scores in full_score_components.items():
    assert scores.shape == (n_samples,), (
        f"{name} shape mismatch: expected {(n_samples,)}, "
        f"found {scores.shape}."
    )

    assert np.isfinite(scores).all(), (
        f"{name} contains NaN or infinite values."
    )

score_stack_full = np.stack(
    list(full_score_components.values()),
    axis=0
).astype(
    np.float32,
    copy=False
)

assert score_stack_full.shape == (
    9,
    n_samples
)

print("Full score-stack shape:", score_stack_full.shape)

In [ ]:
# ===============================
# Full signal flags and agreement
# ===============================

# Spatial signal
S_labels, S_threshold, _ = select_exact_top_fraction(
    S_norm,
    fraction=0.02
)

# Context signal
C_labels, C_threshold, _ = select_exact_top_fraction(
    C_norm,
    fraction=0.02
)

# Convert all previously created labels to Boolean masks.
B_IF_feat = IF_feat_labels.astype(bool)
B_LOF_feat = LOF_feat_labels.astype(bool)
B_MD_feat = MD_feat_labels.astype(bool)

B_IF_emb = IF_emb_labels.astype(bool)
B_LOF_emb = LOF_emb_labels.astype(bool)
B_MD_emb = MD_emb_labels.astype(bool)

B_AE = AE_labels.astype(bool)
B_S = S_labels.astype(bool)
B_C = C_labels.astype(bool)

# Agreement ranges from 0 to 9.
multi_agreement_full = (
    B_IF_feat.astype(np.uint8)
    + B_LOF_feat.astype(np.uint8)
    + B_MD_feat.astype(np.uint8)
    + B_IF_emb.astype(np.uint8)
    + B_LOF_emb.astype(np.uint8)
    + B_MD_emb.astype(np.uint8)
    + B_AE.astype(np.uint8)
    + B_S.astype(np.uint8)
    + B_C.astype(np.uint8)
)

assert multi_agreement_full.shape == (n_samples,)
assert multi_agreement_full.min() >= 0
assert multi_agreement_full.max() <= 9

agreement_values, agreement_counts = np.unique(
    multi_agreement_full,
    return_counts=True
)

print("Full agreement distribution:")

for value, count in zip(
    agreement_values,
    agreement_counts
):
    print(
        f"Agreement {int(value)}: "
        f"{int(count)} pixels"
    )

print("\nSpatial threshold:", S_threshold)
print("Context threshold:", C_threshold)

In [ ]:
# ===============================
# Full multi-signal uncertainty
# ===============================

multi_uncertainty_full = np.std(
    score_stack_full,
    axis=0,
    dtype=np.float32
)

assert multi_uncertainty_full.shape == (n_samples,)
assert np.isfinite(multi_uncertainty_full).all()

full_uncertainty_min = float(
    multi_uncertainty_full.min()
)

full_uncertainty_max = float(
    multi_uncertainty_full.max()
)

if full_uncertainty_max <= full_uncertainty_min:
    raise ValueError(
        "Full uncertainty has no valid normalization range."
    )

multi_uncertainty_full_norm = (
    multi_uncertainty_full
    - full_uncertainty_min
) / (
    full_uncertainty_max
    - full_uncertainty_min
)

multi_uncertainty_full_norm = (
    multi_uncertainty_full_norm.astype(
        np.float32,
        copy=False
    )
)

assert np.isfinite(
    multi_uncertainty_full_norm
).all()

print("Full normalized-uncertainty statistics:")
print(
    "Minimum:",
    float(multi_uncertainty_full_norm.min())
)
print(
    "Maximum:",
    float(multi_uncertainty_full_norm.max())
)
print(
    "Mean:",
    float(multi_uncertainty_full_norm.mean())
)

In [ ]:
# ===============================
# Full reliability score
# ===============================

full_consensus_score = (
    multi_agreement_full.astype(np.float32)
    / 9.0
)

full_uncertainty_safe = np.clip(
    multi_uncertainty_full_norm,
    0.0,
    1.0
)

full_consensus_weight = 0.60
full_certainty_weight = 0.40

R_multi_full = (
    full_consensus_weight
    * full_consensus_score
    +
    full_certainty_weight
    * (1.0 - full_uncertainty_safe)
).astype(
    np.float32,
    copy=False
)

assert R_multi_full.shape == (n_samples,)
assert np.isfinite(R_multi_full).all()
assert R_multi_full.min() >= 0.0
assert R_multi_full.max() <= 1.0

print("Full reliability statistics:")
print("Minimum:", float(R_multi_full.min()))
print("Maximum:", float(R_multi_full.max()))
print("Mean:", float(R_multi_full.mean()))
print(
    "95th percentile:",
    float(np.percentile(R_multi_full, 95))
)
print(
    "99th percentile:",
    float(np.percentile(R_multi_full, 99))
)

In [ ]:
# ===============================
# Full final anomaly score
# ===============================

A_full_pre_reliability = A_spatial_norm

assert A_full_pre_reliability.shape == (n_samples,)
assert np.isfinite(A_full_pre_reliability).all()

A_final_full_raw = (
    A_full_pre_reliability
    * R_multi_full
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(A_final_full_raw).all()

# Robust normalization using original-data percentiles.
full_final_p1 = float(
    np.percentile(
        A_final_full_raw,
        1
    )
)

full_final_p99 = float(
    np.percentile(
        A_final_full_raw,
        99
    )
)

if full_final_p99 <= full_final_p1:
    raise ValueError(
        "Invalid Full final-score normalization range."
    )

A_final_full_norm = (
    np.clip(
        A_final_full_raw,
        full_final_p1,
        full_final_p99
    )
    - full_final_p1
) / (
    full_final_p99
    - full_final_p1
)

A_final_full_norm = A_final_full_norm.astype(
    np.float32,
    copy=False
)

assert np.isfinite(A_final_full_norm).all()
assert A_final_full_norm.min() >= 0.0
assert A_final_full_norm.max() <= 1.0

print("Full final-score statistics:")
print("Minimum:", float(A_final_full_norm.min()))
print("Maximum:", float(A_final_full_norm.max()))
print("Mean:", float(A_final_full_norm.mean()))
print(
    "95th percentile:",
    float(np.percentile(A_final_full_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(A_final_full_norm, 98))
)
print(
    "99th percentile:",
    float(np.percentile(A_final_full_norm, 99))
)

In [ ]:
# ===============================
# Full exact top-2% anomaly selection
# ===============================

full_anomaly_labels, full_anomaly_threshold, _ = (
    select_exact_top_fraction(
        A_final_full_norm,
        fraction=0.02
    )
)

print("Full final threshold:", full_anomaly_threshold)
print(
    "Full anomaly count:",
    int(full_anomaly_labels.sum())
)
print(
    "Full anomaly percentage:",
    float(full_anomaly_labels.mean() * 100)
)

expected_count = int(
    np.ceil(
        0.02 * n_samples
    )
)

assert full_anomaly_labels.sum() == expected_count

In [ ]:
# ===============================
# Stage 7B: Pure variant
# ===============================

# Pure base uses only the AE reconstruction anomaly signal.
A_base_pure = ae_score_norm

pure_base_weight = 0.70
pure_spatial_weight = 0.30

A_spatial_pure = (
    pure_base_weight
    * A_base_pure
    +
    pure_spatial_weight
    * S_norm
).astype(
    np.float32,
    copy=False
)

assert A_spatial_pure.shape == (n_samples,)
assert np.isfinite(A_spatial_pure).all()

# Normalize using original-data percentiles.
pure_spatial_p1 = float(
    np.percentile(
        A_spatial_pure,
        1
    )
)

pure_spatial_p99 = float(
    np.percentile(
        A_spatial_pure,
        99
    )
)

if pure_spatial_p99 <= pure_spatial_p1:
    raise ValueError(
        "Invalid Pure spatial-score normalization range."
    )

A_spatial_pure_norm = (
    np.clip(
        A_spatial_pure,
        pure_spatial_p1,
        pure_spatial_p99
    )
    - pure_spatial_p1
) / (
    pure_spatial_p99
    - pure_spatial_p1
)

A_spatial_pure_norm = (
    A_spatial_pure_norm.astype(
        np.float32,
        copy=False
    )
)

assert np.isfinite(A_spatial_pure_norm).all()

print("Pure spatial-score statistics:")
print("Minimum:", float(A_spatial_pure_norm.min()))
print("Maximum:", float(A_spatial_pure_norm.max()))
print("Mean:", float(A_spatial_pure_norm.mean()))

In [ ]:
# ===============================
# Pure reliability signals
# ===============================

score_stack_pure = np.stack(
    [
        ae_score_norm,
        S_norm,
        C_norm,
    ],
    axis=0
).astype(
    np.float32,
    copy=False
)

assert score_stack_pure.shape == (
    3,
    n_samples
)

print("Pure score-stack shape:", score_stack_pure.shape)

# Reuse exact top-2% labels.
B_AE_pure = AE_labels.astype(bool)
B_S_pure = S_labels.astype(bool)
B_C_pure = C_labels.astype(bool)

multi_agreement_pure = (
    B_AE_pure.astype(np.uint8)
    + B_S_pure.astype(np.uint8)
    + B_C_pure.astype(np.uint8)
)

assert multi_agreement_pure.min() >= 0
assert multi_agreement_pure.max() <= 3

pure_agreement_values, pure_agreement_counts = np.unique(
    multi_agreement_pure,
    return_counts=True
)

print("Pure agreement distribution:")

for value, count in zip(
    pure_agreement_values,
    pure_agreement_counts
):
    print(
        f"Agreement {int(value)}: "
        f"{int(count)} pixels"
    )

In [ ]:
# ===============================
# Pure uncertainty
# ===============================

multi_uncertainty_pure = np.std(
    score_stack_pure,
    axis=0,
    dtype=np.float32
)

assert np.isfinite(multi_uncertainty_pure).all()

pure_uncertainty_min = float(
    multi_uncertainty_pure.min()
)

pure_uncertainty_max = float(
    multi_uncertainty_pure.max()
)

if pure_uncertainty_max <= pure_uncertainty_min:
    raise ValueError(
        "Pure uncertainty has no valid normalization range."
    )

multi_uncertainty_pure_norm = (
    multi_uncertainty_pure
    - pure_uncertainty_min
) / (
    pure_uncertainty_max
    - pure_uncertainty_min
)

multi_uncertainty_pure_norm = (
    multi_uncertainty_pure_norm.astype(
        np.float32,
        copy=False
    )
)

assert np.isfinite(
    multi_uncertainty_pure_norm
).all()

print("Pure normalized-uncertainty statistics:")
print(
    "Minimum:",
    float(multi_uncertainty_pure_norm.min())
)
print(
    "Maximum:",
    float(multi_uncertainty_pure_norm.max())
)
print(
    "Mean:",
    float(multi_uncertainty_pure_norm.mean())
)

In [ ]:
# ===============================
# Pure reliability score
# ===============================

pure_consensus_score = (
    multi_agreement_pure.astype(np.float32)
    / 3.0
)

pure_uncertainty_safe = np.clip(
    multi_uncertainty_pure_norm,
    0.0,
    1.0
)

pure_consensus_weight = 0.60
pure_certainty_weight = 0.40

R_multi_pure = (
    pure_consensus_weight
    * pure_consensus_score
    +
    pure_certainty_weight
    * (1.0 - pure_uncertainty_safe)
).astype(
    np.float32,
    copy=False
)

assert R_multi_pure.shape == (n_samples,)
assert np.isfinite(R_multi_pure).all()
assert R_multi_pure.min() >= 0.0
assert R_multi_pure.max() <= 1.0

print("Pure reliability statistics:")
print("Minimum:", float(R_multi_pure.min()))
print("Maximum:", float(R_multi_pure.max()))
print("Mean:", float(R_multi_pure.mean()))
print(
    "95th percentile:",
    float(np.percentile(R_multi_pure, 95))
)
print(
    "99th percentile:",
    float(np.percentile(R_multi_pure, 99))
)

In [ ]:
# ===============================
# Pure final anomaly score
# ===============================

A_final_pure_raw = (
    A_spatial_pure_norm
    * R_multi_pure
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(A_final_pure_raw).all()

pure_final_p1 = float(
    np.percentile(
        A_final_pure_raw,
        1
    )
)

pure_final_p99 = float(
    np.percentile(
        A_final_pure_raw,
        99
    )
)

if pure_final_p99 <= pure_final_p1:
    raise ValueError(
        "Invalid Pure final-score normalization range."
    )

A_final_pure_norm = (
    np.clip(
        A_final_pure_raw,
        pure_final_p1,
        pure_final_p99
    )
    - pure_final_p1
) / (
    pure_final_p99
    - pure_final_p1
)

A_final_pure_norm = (
    A_final_pure_norm.astype(
        np.float32,
        copy=False
    )
)

assert np.isfinite(A_final_pure_norm).all()
assert A_final_pure_norm.min() >= 0.0
assert A_final_pure_norm.max() <= 1.0

print("Pure final-score statistics:")
print("Minimum:", float(A_final_pure_norm.min()))
print("Maximum:", float(A_final_pure_norm.max()))
print("Mean:", float(A_final_pure_norm.mean()))
print(
    "95th percentile:",
    float(np.percentile(A_final_pure_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(A_final_pure_norm, 98))
)
print(
    "99th percentile:",
    float(np.percentile(A_final_pure_norm, 99))
)

In [ ]:
# ===============================
# Pure exact top-2% anomaly selection
# ===============================

pure_anomaly_labels, pure_anomaly_threshold, _ = (
    select_exact_top_fraction(
        A_final_pure_norm,
        fraction=0.02
    )
)

print("Pure final threshold:", pure_anomaly_threshold)
print(
    "Pure anomaly count:",
    int(pure_anomaly_labels.sum())
)
print(
    "Pure anomaly percentage:",
    float(pure_anomaly_labels.mean() * 100)
)

assert pure_anomaly_labels.sum() == expected_count

In [ ]:
# ===============================
# Save final Full and Pure outputs
# ===============================

np.save(
    "full_final_anomaly_score.npy",
    A_final_full_norm
)

np.save(
    "full_final_anomaly_labels.npy",
    full_anomaly_labels.astype(
        np.uint8,
        copy=False
    )
)

np.save(
    "pure_final_anomaly_score.npy",
    A_final_pure_norm
)

np.save(
    "pure_final_anomaly_labels.npy",
    pure_anomaly_labels.astype(
        np.uint8,
        copy=False
    )
)

print("Saved final mapping outputs:")
print("full_final_anomaly_score.npy")
print("full_final_anomaly_labels.npy")
print("pure_final_anomaly_score.npy")
print("pure_final_anomaly_labels.npy")

In [ ]:
# ===============================
# Save reliability diagnostics
# ===============================

np.savez_compressed(
    "full_pure_reliability_diagnostics.npz",

    full_agreement=multi_agreement_full,
    full_uncertainty=multi_uncertainty_full_norm,
    full_reliability=R_multi_full,

    pure_agreement=multi_agreement_pure,
    pure_uncertainty=multi_uncertainty_pure_norm,
    pure_reliability=R_multi_pure,
)

print("Saved: full_pure_reliability_diagnostics.npz")

In [ ]:
# ===============================
# Save Full and Pure normalization parameters
# ===============================

np.savez(
    "full_pure_final_normalization_params.npz",

    full_uncertainty_min=np.float32(
        full_uncertainty_min
    ),
    full_uncertainty_max=np.float32(
        full_uncertainty_max
    ),
    full_final_p1=np.float32(
        full_final_p1
    ),
    full_final_p99=np.float32(
        full_final_p99
    ),

    pure_spatial_p1=np.float32(
        pure_spatial_p1
    ),
    pure_spatial_p99=np.float32(
        pure_spatial_p99
    ),
    pure_uncertainty_min=np.float32(
        pure_uncertainty_min
    ),
    pure_uncertainty_max=np.float32(
        pure_uncertainty_max
    ),
    pure_final_p1=np.float32(
        pure_final_p1
    ),
    pure_final_p99=np.float32(
        pure_final_p99
    ),

    full_base_weight=np.float32(
        full_base_weight
    ),
    full_spatial_weight=np.float32(
        full_spatial_weight
    ),
    pure_base_weight=np.float32(
        pure_base_weight
    ),
    pure_spatial_weight=np.float32(
        pure_spatial_weight
    ),

    consensus_weight=np.float32(0.60),
    certainty_weight=np.float32(0.40),
)

print(
    "Saved: full_pure_final_normalization_params.npz"
)